# FASE 4 — Anomaly Detection
## Isolation Forest — Mendeteksi Kondisi Abnormal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
plt.style.use('seaborn-v0_8-darkgrid')
df = pd.read_csv('../data/raw_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
features = ['temperature_c', 'vibration_mm_s', 'current_a']
scaler = StandardScaler()
X = scaler.fit_transform(df[features])
iso = IsolationForest(contamination=0.02, random_state=42)
df['anomaly'] = iso.fit_predict(X)
df['anomaly_flag'] = (df['anomaly'] == -1).astype(int)
print(f'Total anomali: {df["anomaly_flag"].sum()} ({df["anomaly_flag"].mean()*100:.1f}%)')
print('\nAnomalies per machine:')
print(df.groupby('machine_id')['anomaly_flag'].sum())

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,6))
normal = df[df['anomaly_flag']==0]
anom   = df[df['anomaly_flag']==1]
axes[0].scatter(normal['temperature_c'], normal['vibration_mm_s'], c='#2196F3', alpha=0.3, s=8, label=f'Normal ({len(normal):,})')
axes[0].scatter(anom['temperature_c'],   anom['vibration_mm_s'],   c='#F44336', alpha=0.8, s=30, marker='X', label=f'Anomaly ({len(anom):,})')
axes[0].set_xlabel('Temperature (C)'); axes[0].set_ylabel('Vibration (mm/s)')
axes[0].set_title('Anomaly Detection: Temperature vs Vibration', fontweight='bold')
axes[0].legend()
axes[1].scatter(normal['current_a'], normal['vibration_mm_s'], c='#2196F3', alpha=0.3, s=8, label='Normal')
axes[1].scatter(anom['current_a'],   anom['vibration_mm_s'],   c='#F44336', alpha=0.8, s=30, marker='X', label='Anomaly')
axes[1].set_xlabel('Current (A)'); axes[1].set_ylabel('Vibration (mm/s)')
axes[1].set_title('Anomaly Detection: Current vs Vibration', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('../data/plots/anomaly_detail.png', dpi=150)
plt.show()

In [ ]:
# Show sample anomalies
print('Sample Anomaly Records:')
print(df[df['anomaly_flag']==1][['timestamp','machine_id','temperature_c','vibration_mm_s','current_a']].head(10).to_string())

## Anomaly Detection Results

- **Total Anomali:** 876 records (2% dari 43,800)
- **Metode:** Isolation Forest (contamination=2%)
- **Input Features:** temperature_c, vibration_mm_s, current_a
- **Karakteristik:** Suhu > 72°C DAN vibrasi > 3.8 mm/s
- **Tindakan:** Alert otomatis + schedule inspection